###A complete network in a OOP pipeline

In [32]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

#creating dataset
class temperature(Dataset):
  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]


In [33]:

#creating the layers and mapping via forward()
class Net(nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim):
   super().__init__()
   self.fc1 = nn.Linear(input_dim, hidden_dim)
   self.fc2 = nn.Linear(hidden_dim, hidden_dim)
   self.fc3 = nn.Linear(hidden_dim, output_dim)

  def forward(self, x):
    x = torch.relu(self.fc1(x))
    x = torch.relu(self.fc2(x))
    x = self.fc3(x)
    return x


In [34]:
#the heart part - training
class training:
  def __init__(self, model, criterion, optimizer, device):
    self.model = model
    self.criterion = criterion
    self.optimizer = optimizer
    self.device = device

  def training_epochs(self, loader):
    self.model.train()
    total_loss = 0.0
    for data, target in loader:
      data, target = data.to(self.device), target.to(self.device)
      self.optimizer.zero_grad()
      output = self.model(data)
      loss = self.criterion(output, target.view(-1, 1)) # Reshape target to match output shape
      loss.backward()
      self.optimizer.step()
      total_loss += loss.item()
    return total_loss / len(loader)

  def evaluate(self, loader):
    self.model.eval()
    total_loss = 0.0
    with torch.no_grad():
      for data, target in loader:
        data = data.to(self.device)
        target = target.to(self.device)
        output = self.model(data)
        loss = self.criterion(output, target.view(-1, 1))
        total_loss += loss.item()
        avg_loss = total_loss / len(loader)
    return avg_loss

  def fit(self, train_loader, val_loader, epochs):
    train_losses = []
    val_losses = []
    for epoch in range(epochs):
      train_loss = self.training_epochs(train_loader)
      val_loss = self.evaluate(val_loader)
      train_losses.append(train_loss)
      val_losses.append(val_loss)
      if (epoch + 1) % 10 == 0:
          print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

In [35]:
#executing the main
#create tensors
data_X = torch.randn(500, 5).to(torch.float32)
data_y = torch.randn(500,).to(torch.float32)

#data splitting
dataset = temperature(data_X, data_y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

#data loader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

#powersupply to our model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Net(input_dim=5, hidden_dim=64, output_dim=1).to(device)
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.01)

#model feeding via fit
trainer = training(model, criterion, optimizer, device)
trainer.fit(train_loader, val_loader, epochs=50)

Epoch  10 | Train Loss: 0.9007 | Val Loss: 1.0996
Epoch  20 | Train Loss: 0.8758 | Val Loss: 1.0916
Epoch  30 | Train Loss: 0.8299 | Val Loss: 1.0757
Epoch  40 | Train Loss: 0.7962 | Val Loss: 1.0759
Epoch  50 | Train Loss: 0.7721 | Val Loss: 1.0743


<i><h8>The validation loss is quite high more like to be overfitting because of random data generation</h8></i>